# Lab 13 - Evaluate the live Foundry IQ RAG path

## Does the answer stay good across repeated runs?

Lab 12 queried Search directly. Here, the **Foundry IQ knowledge base** plans and retrieves the evidence:

```text
question -> Foundry IQ -> Azure AI Search -> evidence and references
         -> model creates a cited answer
```

Use `extractiveData` mode to get passages and source references, not an IQ-written answer. A separate model writes the answer so you can inspect retrieval and generation independently.

Run each WHO question three times, then score each saved answer several times. This separates changes in the live system from changes in evaluator scores.

## Before you start

Complete Lab 3 and configure the Foundry project, model, Search endpoint and knowledge base. The knowledge source defaults to `who-guidelines-ks`. You need Foundry User and Search Index Data Reader access.

Replace each `...` blank before running its cell.

> Three topics and three repeats demonstrate the method, not production reliability.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0"

## 0. Connect and load the shared helpers

Use the same benchmark helpers as Lab 12 so both paths calculate quality, timing and usage consistently.

The IQ **retrieve action** calls the knowledge-base endpoint using your Azure CLI identity. A project-scoped OpenAI client generates answers and submits evaluations.

`API_VERSION` selects the preview retrieve features. Choose live repeats later; `JUDGE_REPEATS` defaults to three scores per saved answer.

**You should see** the knowledge base, source, API version, answer model, evaluator model and judge repeat count.

In [ ]:
import json
import os
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path
from statistics import median
from urllib.parse import quote
from uuid import uuid4

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    helper_path = folder / "labs" / "day_2" / "evaluation_utils.py"
    if helper_path.exists():
        REPO_ROOT = folder
        sys.path.insert(0, str(helper_path.parent))
        break
else:
    raise FileNotFoundError("labs/day_2/evaluation_utils.py was not found.")

from evaluation_utils import (
    answer_json_schema,
    context_signal_coverage,
    document_evidence_keys,
    estimate_cost_usd,
    format_iq_context,
    iq_activity_metrics,
    load_cases,
    percentile_95,
    price_rates_from_env,
    primitive,
    render_cited_answer,
    resolve_iq_references,
    response_token_usage,
    retrieval_recall,
    validate_cited_answer,
)
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")
EVALUATOR_MODEL = os.getenv("RAG_EVALUATOR_MODEL", "") or MODEL_DEPLOYMENT
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "").rstrip("/")
KNOWLEDGE_BASE = os.getenv("AZURE_SEARCH_KNOWLEDGE_BASE", "")
KNOWLEDGE_SOURCE = os.getenv("AZURE_SEARCH_KNOWLEDGE_SOURCE", "who-guidelines-ks")
API_VERSION = "2026-08-01-preview"
JUDGE_REPEATS = int(os.getenv("RAG_EVAL_JUDGE_REPEATS", "3"))
if not 3 <= JUDGE_REPEATS <= 9 or JUDGE_REPEATS % 2 == 0:
    raise ValueError("RAG_EVAL_JUDGE_REPEATS must be an odd number from 3 to 9.")
missing = [
    name
    for name, value in {
        "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT,
        "AZURE_SEARCH_ENDPOINT": SEARCH_ENDPOINT,
        "AZURE_SEARCH_KNOWLEDGE_BASE": KNOWLEDGE_BASE,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Set these before continuing: {', '.join(missing)}")


def check_todos(**answers: object) -> None:
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


def wait_for_run(eval_id, run, expected_items):
    deadline = time.monotonic() + 1200
    while run.status not in ("completed", "failed", "canceled"):
        if time.monotonic() > deadline:
            raise TimeoutError("Evaluation exceeded 20 minutes.")
        time.sleep(5)
        run = client.evals.runs.retrieve(run_id=run.id, eval_id=eval_id)
        print("status:", run.status)
    items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    if run.status != "completed":
        raise RuntimeError(f"Evaluation ended as {run.status}: {primitive(getattr(run, 'error', None))}")
    item_deadline = time.monotonic() + 120
    while len(items) < expected_items:
        if time.monotonic() > item_deadline:
            raise TimeoutError(f"Only {len(items)}/{expected_items} output items became visible.")
        time.sleep(2)
        items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    return run, items


SUFFIX = uuid4().hex[:8]
credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=300, max_retries=0)
RATES = price_rates_from_env()
RETRIEVE_URL = (
    f"{SEARCH_ENDPOINT}/knowledgebases/{quote(KNOWLEDGE_BASE, safe='')}"
    f"/retrieve?api-version={API_VERSION}"
)
print({
    "knowledge_base": KNOWLEDGE_BASE,
    "knowledge_source": KNOWLEDGE_SOURCE,
    "api_version": API_VERSION,
    "judge_repeats": JUDGE_REPEATS,
    "generator": MODEL_DEPLOYMENT,
    "judge": EVALUATOR_MODEL,
})

## 1. Load the matching benchmark

Use the same three WHO topics and answer rules as Lab 12.

Each case adds a `knowledge_source_filter` on `publication_id`. This restricts IQ to the known publication so you can study passage selection within it. It does **not** test whether an unfiltered query would choose the right publication.

IQ receives the question and filter, not expected answers. The separate answer model later receives the required answer parts, as in Lab 12.

**You should see** dataset and answer-contract versions plus three IQ case IDs.

In [ ]:
DATASET_PATH = REPO_ROOT / "labs" / "data" / "medical_rag_evaluation_cases.json"
DATASET, IQ_CASES = load_cases(DATASET_PATH, "foundry_iq")
assert DATASET["dataset_id"] == "umc-who-rag-evaluation-v1"
assert len(IQ_CASES) == 3
assert all(case["knowledge_source_filter"].startswith("publication_id eq") for case in IQ_CASES)
print({
    "dataset": DATASET["dataset_id"],
    "answer_contract": DATASET["answer_contract_version"],
    "cases": [case["case_id"] for case in IQ_CASES],
})

### To-Do 1 - Return evidence and repeat retrieval

Set:

- `OUTPUT_MODE = "extractiveData"` to return selected passages, source references and activity records.
- `RESPONSE_REPEATS = 3` to run each question three independent times.

Separate evidence from answers so you can see where a failure begins. Repeated calls reveal changes in passages, answers, timing and usage.

**Predict:** if an IQ-written answer became the only context for another model, why would errors be harder to trace?

<details><summary>Show solution code</summary>

```python
OUTPUT_MODE = "extractiveData"
RESPONSE_REPEATS = 3
```

</details>

In [ ]:
OUTPUT_MODE = ...  # TODO 1: inspectable knowledge-base output mode.
RESPONSE_REPEATS = ...  # TODO 1: independent live runs per benchmark case.
check_todos(OUTPUT_MODE=OUTPUT_MODE, RESPONSE_REPEATS=RESPONSE_REPEATS)
assert OUTPUT_MODE == "extractiveData"
assert RESPONSE_REPEATS == 3
print({"output_mode": OUTPUT_MODE, "response_repeats": RESPONSE_REPEATS})

## 2. Retrieve evidence and generate nine answers

The first cell defines the helpers; the second runs three cases three times.

For each run, the helpers:

1. Send IQ the question and publication filter.
2. Match each returned `ref_id` to its passage and build the answer context.
3. Measure publication recall and look for required context phrases.
4. Generate JSON claims citing only returned source IDs.
5. Check answer parts and citations; record timing, usage and errors.

IQ's **activity records** describe planning, Search calls, timing and warnings. `failOnError: true` prevents a source failure from being silently accepted. HTTP 206 means retrieval was only partly completed; keep partial results and warnings visible.

**You should see** nine records with request IDs, references, activity, metrics and cited answers, followed by `PASS`.

In [ ]:
def retrieve_from_iq(case):
    client_request_id = str(uuid4())
    body = {
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": case["query"]}]}
        ],
        "knowledgeSourceParams": [
            {
                "knowledgeSourceName": KNOWLEDGE_SOURCE,
                "kind": "searchIndex",
                "alwaysQuerySource": True,
                "failOnError": True,
                "includeReferences": True,
                "includeReferenceSourceData": True,
                "filterAddOn": case["knowledge_source_filter"],
                "rerankerThreshold": 2.0,
                "maxOutputDocuments": 50,
            }
        ],
        "includeActivity": True,
        "outputMode": OUTPUT_MODE,
        "retrievalReasoningEffort": {"kind": "low"},
        "maxOutputDocuments": 12,
        "maxOutputSize": 20000,
    }
    token = credential.get_token("https://search.azure.com/.default").token
    request = urllib.request.Request(
        RETRIEVE_URL,
        data=json.dumps(body).encode("utf-8"),
        method="POST",
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
            "x-ms-client-request-id": client_request_id,
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=240) as response:
            payload = json.load(response)
            payload["_client_request_id"] = client_request_id
            return response.status, payload
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"IQ retrieve failed with HTTP {error.code}; request {client_request_id}: {detail[:2000]}"
        ) from error


def generate_cited_iq_answer(case, context, valid_reference_ids):
    required_parts = case["required_answer_parts"]
    part_ids = [part["id"] for part in required_parts]
    parts_text = "\n".join(
        f"- {part['id']}: {part['description']}" for part in required_parts
    )
    schema = answer_json_schema(part_ids, valid_reference_ids)
    response = client.responses.create(
        model=MODEL_DEPLOYMENT,
        instructions=(
            "Answer only from the supplied Foundry IQ evidence. Cover each required answer part "
            "with one or more concise factual claims. Give every claim exactly one answer_part_id "
            "and one or more returned source_ids. Put a part in insufficient_evidence only when "
            "the supplied evidence cannot support it. Never invent a reference or give patient-specific advice."
        ),
        input=(
            f"Question: {case['query']}\n\n"
            f"Required answer parts ({DATASET['answer_contract_version']}):\n{parts_text}\n\n"
            f"Foundry IQ evidence:\n{context}"
        ),
        text={
            "format": {
                "type": "json_schema",
                "name": "medical_iq_answer",
                "strict": True,
                "schema": schema,
            }
        },
        max_output_tokens=1400,
    )
    validation = validate_cited_answer(
        json.loads(response.output_text),
        valid_reference_ids,
        part_ids,
    )
    return response, validation, render_cited_answer(validation)


print("Foundry IQ retrieval and cited-answer helpers ready.")

In [ ]:
IQ_ROWS = []
for case in IQ_CASES:
    for repeat in range(1, RESPONSE_REPEATS + 1):
        run_case_id = f"{case['case_id']}-R{repeat}"
        total_started = time.perf_counter()
        retrieval_started = time.perf_counter()
        status_code, payload = retrieve_from_iq(case)
        retrieval_ms = (time.perf_counter() - retrieval_started) * 1000

        references = resolve_iq_references(payload)
        if not references:
            raise RuntimeError(f"{run_case_id}: IQ returned no resolved references.")
        context = format_iq_context(references)
        retrieved_keys = set().union(
            *(document_evidence_keys(reference["document"]) for reference in references)
        )
        recall = retrieval_recall(case["expected_evidence_groups"], retrieved_keys)
        signal_result = context_signal_coverage(case["required_context_signals"], context)
        valid_reference_ids = [f"ref_id:{reference['ref_id']}" for reference in references]

        generation_started = time.perf_counter()
        response, citations, answer = generate_cited_iq_answer(
            case, context, valid_reference_ids
        )
        generation_ms = (time.perf_counter() - generation_started) * 1000

        activity = iq_activity_metrics(payload.get("activity") or [])
        generation_usage = response_token_usage(response)
        for metric in (
            "model_input_tokens",
            "model_output_tokens",
            "agentic_retrieval_tokens",
            "semantic_requests",
        ):
            activity[metric] += generation_usage[metric]
        cost = estimate_cost_usd(activity, RATES)
        source_warnings = [
            {
                "source": item.get("knowledgeSourceName"),
                "warning": item.get("warning"),
                "error": item.get("error"),
            }
            for item in payload.get("activity") or []
            if item.get("warning") or item.get("error")
        ]

        row = {
            "case_id": run_case_id,
            "benchmark_case_id": case["case_id"],
            "repeat": repeat,
            "query": case["query"],
            "ground_truth": case["ground_truth"],
            "context": context,
            "response": answer,
            "http_status": status_code,
            "client_request_id": payload["_client_request_id"],
            "references": len(references),
            "reference_ids": valid_reference_ids,
            "activity_types": [item.get("type") for item in payload.get("activity") or []],
            "retrieval_recall_at_12": recall["recall"],
            "recall_details": recall["groups"],
            "context_signal_coverage": signal_result["coverage"],
            "context_signal_details": signal_result["signals"],
            "answer_part_coverage": citations["answer_part_coverage"],
            "insufficient_evidence": citations["insufficient_evidence"],
            "citation_coverage": citations["citation_coverage"],
            "citation_validity": citations["citation_validity"],
            "source_warnings": source_warnings,
            "retrieval_ms": round(retrieval_ms, 2),
            "generation_ms": round(generation_ms, 2),
            "total_ms": round((time.perf_counter() - total_started) * 1000, 2),
            **activity,
            **cost,
        }
        IQ_ROWS.append(row)
        print(json.dumps({
            key: row[key]
            for key in (
                "case_id",
                "client_request_id",
                "http_status",
                "references",
                "reference_ids",
                "activity_types",
                "retrieval_recall_at_12",
                "context_signal_coverage",
                "answer_part_coverage",
                "citation_coverage",
                "citation_validity",
                "source_warnings",
                "retrieval_ms",
                "generation_ms",
                "query_planning_ms",
                "search_execution_ms",
                "model_input_tokens",
                "model_output_tokens",
                "agentic_retrieval_tokens",
                "semantic_requests",
                "estimated_cost_usd",
            )
        }, indent=2))
        print(answer, "\n")

assert len(IQ_ROWS) == len(IQ_CASES) * RESPONSE_REPEATS
print("PASS - every IQ case completed every live repeat with resolved evidence and strict citations.")

## 3. Score each saved answer several times

You now have three live results per topic: `R1`, `R2` and `R3`. Score each answer three times without changing it:

- **Groundedness** checks it against the evidence from that live run.
- **Response Completeness** checks it against the independent reference answer.

| Repeat | What it reveals |
|---|---|
| Live runs `R1`-`R3` | Changes in retrieval, references, answers, timing and usage |
| Judge runs `J1`-`J3` | Different scores or reasons for the same saved answer |

Keep every score and reason, then use the middle score, the **median**, for each metric. Do not regenerate answers until a judge approves them.

Nine live results scored three times produce 27 evaluation items. Judge token use is reported separately from live-path usage.

**You should see** raw scores and reasons, medians and a report URL.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

eval_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "eval_item_id": {"type": "string"},
            "run_case_id": {"type": "string"},
            "judge_repeat": {"type": "integer"},
            "query": {"type": "string"},
            "context": {"type": "string"},
            "response": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": [
            "eval_item_id",
            "run_case_id",
            "judge_repeat",
            "query",
            "context",
            "response",
            "ground_truth",
        ],
        "additionalProperties": False,
    },
    include_sample_schema=True,
)
criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="groundedness",
        evaluator_name="builtin.groundedness",
        initialization_parameters={"deployment_name": EVALUATOR_MODEL},
        data_mapping={
            "query": "{{item.query}}",
            "context": "{{item.context}}",
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="response_completeness",
        evaluator_name="builtin.response_completeness",
        initialization_parameters={"deployment_name": EVALUATOR_MODEL},
        data_mapping={
            "ground_truth": "{{item.ground_truth}}",
            "response": "{{item.response}}",
        },
    ),
]
evaluation = client.evals.create(
    name=f"day2-iq-rag-eval-{SUFFIX}",
    data_source_config=eval_config,
    testing_criteria=criteria,
)
eval_items = [
    {
        "item": {
            "eval_item_id": f"{row['case_id']}-J{judge_repeat}",
            "run_case_id": row["case_id"],
            "judge_repeat": judge_repeat,
            **{key: row[key] for key in ("query", "context", "response", "ground_truth")},
        }
    }
    for row in IQ_ROWS
    for judge_repeat in range(1, JUDGE_REPEATS + 1)
]
eval_run = client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"day2-iq-rag-run-{SUFFIX}",
    metadata={
        "dataset": DATASET["dataset_id"],
        "answer_contract": DATASET["answer_contract_version"],
        "target": "foundry-iq-extractive-data",
        "response_repeats": str(RESPONSE_REPEATS),
        "judge_repeats": str(JUDGE_REPEATS),
    },
    data_source={
        "type": "jsonl",
        "source": {"type": "file_content", "content": eval_items},
    },
)
print({"evaluation_id": evaluation.id, "run_id": eval_run.id, "items": len(eval_items)})

In [ ]:
eval_run, output_items = wait_for_run(evaluation.id, eval_run, len(eval_items))
METRICS = ("groundedness", "response_completeness")
scores = {
    row["case_id"]: {metric: {} for metric in METRICS}
    for row in IQ_ROWS
}
reasons = {
    row["case_id"]: {metric: {} for metric in METRICS}
    for row in IQ_ROWS
}
for item in output_items:
    data = primitive(item)
    source = data["datasource_item"]
    case_id = source["run_case_id"]
    judge_repeat = int(source["judge_repeat"])
    for metric in METRICS:
        result = next(result for result in data["results"] if result["name"] == metric)
        if result.get("error") or result.get("status") in ("failed", "error", "canceled"):
            raise RuntimeError(f"{case_id}-J{judge_repeat} {metric} failed: {result}")
        reason = str(result.get("reason") or "").strip()
        if not reason:
            raise RuntimeError(f"{case_id}-J{judge_repeat} {metric} returned no reason.")
        scores[case_id][metric][judge_repeat] = float(result["score"])
        reasons[case_id][metric][judge_repeat] = reason

for row in IQ_ROWS:
    for metric in METRICS:
        samples = [
            scores[row["case_id"]][metric][repeat]
            for repeat in range(1, JUDGE_REPEATS + 1)
        ]
        row[f"{metric}_samples"] = samples
        row[f"{metric}_reasons"] = [
            reasons[row["case_id"]][metric][repeat]
            for repeat in range(1, JUDGE_REPEATS + 1)
        ]
        row[f"{metric}_mean"] = round(sum(samples) / len(samples), 3)
        row[metric] = float(median(samples))

print({
    "row_consensus": {
        row["case_id"]: {
            metric: {
                "samples": row[f"{metric}_samples"],
                "median": row[metric],
                "reasons": row[f"{metric}_reasons"],
            }
            for metric in METRICS
        }
        for row in IQ_ROWS
    },
    "evaluator_usage": [primitive(item) for item in (getattr(eval_run, "per_model_usage", None) or [])],
    "report_url": getattr(eval_run, "report_url", None),
})

### To-Do 2 - Require every repeat to pass

Set `MIN_GROUNDEDNESS` and `MIN_COMPLETENESS` to `4.0` on the 1-to-5 scale before reading the summary.

Each live result must have:

- HTTP 200, at least one reference and no source warning.
- Full publication recall, context signals, answer parts and valid citations, with no unsupported required part.
- Median Groundedness and Completeness of at least 4.

A topic gets `ship` only if **all three repeats** pass. The summary keeps the lowest quality scores rather than averaging away a failure.

It also reports **p95 latency**: the duration at or below which 95% of observed runs fall. With three repeats, this is the slowest run, not a production latency estimate.

`judge_disagreement_runs` counts answers that received different scores across judges. Inspect the reasons; disagreement alone is not a failure.

<details><summary>Show solution code</summary>

```python
MIN_GROUNDEDNESS = 4.0
MIN_COMPLETENESS = 4.0
```

</details>

In [ ]:
MIN_GROUNDEDNESS = ...  # TODO 2: median groundedness threshold.
MIN_COMPLETENESS = ...  # TODO 2: median response-completeness threshold.
check_todos(MIN_GROUNDEDNESS=MIN_GROUNDEDNESS, MIN_COMPLETENESS=MIN_COMPLETENESS)
assert MIN_GROUNDEDNESS == MIN_COMPLETENESS == 4.0

for row in IQ_ROWS:
    row["deterministic_gate_passed"] = all([
        row["retrieval_recall_at_12"] == 1.0,
        row["context_signal_coverage"] == 1.0,
        row["answer_part_coverage"] == 1.0,
        not row["insufficient_evidence"],
        row["citation_coverage"] == 1.0,
        row["citation_validity"] == 1.0,
    ])
    row["operational_gate_passed"] = (
        row["http_status"] == 200
        and row["references"] > 0
        and not row["source_warnings"]
    )
    row["semantic_gate_passed"] = (
        row["groundedness"] >= MIN_GROUNDEDNESS
        and row["response_completeness"] >= MIN_COMPLETENESS
    )
    row["release_gate_passed"] = all([
        row["deterministic_gate_passed"],
        row["operational_gate_passed"],
        row["semantic_gate_passed"],
    ])
    row["decision"] = "ship" if row["release_gate_passed"] else "mitigate"

CASE_SUMMARIES = {}
for case in IQ_CASES:
    rows = [row for row in IQ_ROWS if row["benchmark_case_id"] == case["case_id"]]
    CASE_SUMMARIES[case["case_id"]] = {
        "runs": len(rows),
        "min_retrieval_recall": min(row["retrieval_recall_at_12"] for row in rows),
        "min_context_signal_coverage": min(row["context_signal_coverage"] for row in rows),
        "min_citation_validity": min(row["citation_validity"] for row in rows),
        "min_groundedness_median": min(row["groundedness"] for row in rows),
        "min_completeness_median": min(row["response_completeness"] for row in rows),
        "judge_disagreement_runs": sum(
            len(set(row["groundedness_samples"])) > 1
            or len(set(row["response_completeness_samples"])) > 1
            for row in rows
        ),
        "p95_total_ms": percentile_95(row["total_ms"] for row in rows),
        "model_tokens": sum(
            row["model_input_tokens"] + row["model_output_tokens"] for row in rows
        ),
        "agentic_retrieval_tokens": sum(row["agentic_retrieval_tokens"] for row in rows),
        "semantic_requests": sum(row["semantic_requests"] for row in rows),
        "all_release_gates_passed": all(row["release_gate_passed"] for row in rows),
        "decision": "ship" if all(row["release_gate_passed"] for row in rows) else "mitigate",
    }

for row in IQ_ROWS:
    print(json.dumps({
        "case_id": row["case_id"],
        "decision": row["decision"],
        "http_status": row["http_status"],
        "source_warnings": row["source_warnings"],
        "retrieval_recall_at_12": row["retrieval_recall_at_12"],
        "context_signal_coverage": row["context_signal_coverage"],
        "citation_coverage": row["citation_coverage"],
        "citation_validity": row["citation_validity"],
        "groundedness": {
            "samples": row["groundedness_samples"],
            "median": row["groundedness"],
            "reasons": row["groundedness_reasons"],
        },
        "response_completeness": {
            "samples": row["response_completeness_samples"],
            "median": row["response_completeness"],
            "reasons": row["response_completeness_reasons"],
        },
        "latency_ms": {
            "iq_retrieval": row["retrieval_ms"],
            "generation": row["generation_ms"],
            "total": row["total_ms"],
            "query_planning": row["query_planning_ms"],
            "search": row["search_execution_ms"],
        },
        "usage": {
            "model_input_tokens": row["model_input_tokens"],
            "model_output_tokens": row["model_output_tokens"],
            "agentic_retrieval_tokens": row["agentic_retrieval_tokens"],
            "semantic_requests": row["semantic_requests"],
        },
        "estimated_cost_usd": row["estimated_cost_usd"],
    }, indent=2))
print(json.dumps({"case_summaries": CASE_SUMMARIES}, indent=2))

## Confirm that the repeated evaluation completed

The final check requires nine live results, 27 judge items, valid metrics, repeated reasons, timing and a three-run summary for each topic.

A `PASS` message means the evidence is ready to inspect. It does not override a `mitigate` decision or force every answer to pass.

**You should see** `PASS` and one decision per WHO topic.

In [ ]:
assert eval_run.status == "completed"
assert len(IQ_ROWS) == len(IQ_CASES) * RESPONSE_REPEATS == 9
assert len(output_items) == len(IQ_ROWS) * JUDGE_REPEATS
assert all(row["references"] > 0 and row["context"] and row["response"] for row in IQ_ROWS)
assert all(0.0 <= row["retrieval_recall_at_12"] <= 1.0 for row in IQ_ROWS)
assert all(0.0 <= row["citation_validity"] <= 1.0 for row in IQ_ROWS)
assert all(len(row["groundedness_reasons"]) == JUDGE_REPEATS for row in IQ_ROWS)
assert all(row["retrieval_ms"] > 0 and row["total_ms"] >= row["retrieval_ms"] for row in IQ_ROWS)
assert set(CASE_SUMMARIES) == {case["case_id"] for case in IQ_CASES}
assert all(summary["runs"] == RESPONSE_REPEATS for summary in CASE_SUMMARIES.values())
print("PASS - repeated IQ retrieval, citation, judging, activity and worst-case evidence are populated.")
print("Decisions:", {case_id: summary["decision"] for case_id, summary in CASE_SUMMARIES.items()})

## Find the first failing step

| Failure | Inspect |
|---|---|
| HTTP 206, warning or source error | IQ retrieval and source availability |
| Publication found but context signals missing | Selected passages |
| Context signals pass but answer parts are missing | Exact evidence and generation; keyword matches do not prove enough evidence |
| Valid references but low Groundedness | Whether claims match their sources |
| One repeat fails | That run's request ID and activity |
| High planning time, Search calls or latency | IQ reasoning, filters and source design |

Keep failed repeats visible. A good average cannot cancel one.

<details><summary>Optional cost estimates</summary>

Tokens and requests are always counted. USD estimates require rates from your applicable price sheet; they are not Azure billing records.

</details>

## What you learned

- `extractiveData` lets you inspect retrieved passages separately from the generated answer.
- Live repeats measure system changes; judge repeats measure scoring changes.
- Use the weakest result and slow-run timing, not just averages, to assess reliability.

A returned `ref_id` identifies a source only within that response. It is not a permanent document key.

**Check your understanding**

1. Why does a valid `ref_id` not prove a claim is supported?
2. Two repeats pass and one returns HTTP 206. Can the topic ship?
3. IQ uses more tokens but gives more complete answers. Is that automatically better?

<details><summary>Compare your answers</summary>

1. It identifies returned evidence, not whether the claim represents it accurately.
2. No. Every repeat must complete without source warnings under this policy.
3. Compare the quality gain with latency, cost and requirements.

</details>

Further reading: [knowledge-base retrieval](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve), [Foundry IQ](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq), and [RAG evaluators](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/rag-evaluators).

**Expected artifact:** nine live results, 27 judge items, three topic summaries and one report URL.

**Finish:** close local clients. The report remains in Foundry.

**Next:** apply these benchmarks, quality checks and telemetry patterns in Day 3.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed local clients. The evaluation report remains in Foundry.")